In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. Модель «структура — свойство»

Дескрипторы можно объединить в модель свойства, но небольшое число лекарственных объектов не образует обучающую выборку. Вернёмся к ESOL как отдельному опубликованному контролю QSPR. Гипотеза: линейная модель дескрипторов превосходит среднее обучающей выборки на отложенных молекулах. Вывод о растворимости A/B потребует проверки области применимости.

Из всех ESOL SMILES вычислите MolWt, MolLogP, TPSA, HBD, HBA. Группируйте по каноническому isomeric SMILES, используйте GroupShuffleSplit с seed=42 и test_size=.2. Сравните Ridge pipeline с DummyRegressor. Сохраните ID train/test и проверьте отсутствие пересечения групп. Сделайте диаграмму y–prediction.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M12-L01','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors,Lipinski
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
rows=esol();mols=[Chem.MolFromSmiles(r['smiles']) for r in rows];assert all(m is not None for m in mols)
def features(m):return [Descriptors.MolWt(m),Descriptors.MolLogP(m),Descriptors.TPSA(m),Lipinski.NumHDonors(m),Lipinski.NumHAcceptors(m)]
X=np.array([features(m) for m in mols]);y=np.array([r['measured'] for r in rows])
groups=np.array([Chem.MolToSmiles(m,isomericSmiles=True) for m in mols])
train,test=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(X,y,groups))
assert not set(groups[train])&set(groups[test])


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
model=make_pipeline(StandardScaler(),Ridge(alpha=1));model.fit(X[train],y[train]);pred=model.predict(X[test])
baseline=DummyRegressor().fit(X[train],y[train]).predict(X[test])
rmse=lambda a,b:float(np.sqrt(np.mean((a-b)**2)))
print({'Ridge_RMSE':rmse(y[test],pred),'baseline_RMSE':rmse(y[test],baseline)})
fig,ax=plt.subplots();ax.scatter(y[test],pred,s=10);ax.plot([-10,3],[-10,3],color='gray')
ax.set(xlabel='Measured logS',ylabel='Held-out predicted logS');publish_plot(fig,'qspr.png')


{'Ridge_RMSE': 1.0133990883867618, 'baseline_RMSE': 2.0465800881298293}


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'kind':'calculated','seed':42,'split':'identity groups','train_ids':train.tolist(),'test_ids':test.tolist(),
        'test_RMSE_logS':rmse(y[test],pred),'baseline_RMSE_logS':rmse(y[test],baseline),'B_prediction':'NOT_VALIDATED'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M12-L01','A',result,ROOT,inputs=['practicuum/shared/data/esol.csv'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261118-full-course-windows\workspace\my_research\evidence\M12-L01\20261118T141636025776Z-A-d08d36e5eb404643bc49af1d6e5cf154.json


Даже улучшение над baseline не доказывает причинность дескрипторов или универсальный перенос. Не включайте опубликованный ESOL predicted в признаки: это чужая модель с неизвестной относительно вашего split историей. Далее проверим удалённость A и B от train.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?